# Correctness Eval Results Comparison and Analysis

Load one or more local `CorrectnessEvalResult` pickles and compare guardrail performance side-by-side.

Use `RESULT_PATHS` for the shared comparison cells, and optionally set `MAIN_TARGET_PATH`
to focus the single-result deep dives (ROC/PR, category breakdowns, difficulty, cost, attempt browser).


In [ ]:
import pathlib
import typing

import matplotlib.pyplot as plt
import pandas as pd

import pyine.evals.correctness
import pyine.evals.correctness.analysis
import pyine.evals.persistence

In [ ]:
TARGET_EVAL_SUBSET_NAME = "guardrail_test"
TARGET_FPR = 0.01

# set if the result evaluates multiple guardrail types; None for single-type evals
GUARDRAIL_TYPE_NAME: str | None = None

# one or more local pickle paths to CorrectnessEvalResult artifacts
RESULT_PATHS: list[str] = [
    # "logs/evals/guardrail_a.pkl",
    # "logs/evals/guardrail_b.pkl",
]

# optional labels aligned with RESULT_PATHS; defaults to the shortest unique path suffix when omitted
DISPLAY_NAMES: list[str] | None = None

# optional: choose one of RESULT_PATHS for single-result deep dives
# defaults to RESULT_PATHS[0] when omitted
MAIN_TARGET_PATH: str | None = None

In [ ]:
def _normalize_result_path(
    raw_path: str,
) -> pathlib.Path:
    return pathlib.Path(raw_path).expanduser().resolve()


def _build_short_unique_path_labels(
    result_paths: list[pathlib.Path],
) -> list[str]:
    if not result_paths:
        return []
    path_parts_per_result = []
    for result_path in result_paths:
        path_parts = list(result_path.parts)
        path_parts[-1] = result_path.stem
        path_parts_per_result.append(path_parts)
    max_part_count = max(len(path_parts) for path_parts in path_parts_per_result)
    for suffix_length in range(1, max_part_count + 1):
        candidate_labels = ["/".join(path_parts[-suffix_length:]) for path_parts in path_parts_per_result]
        if len(candidate_labels) == len(set(candidate_labels)):
            return candidate_labels
    return [f"{'/'.join(path_parts)}#{path_idx}" for path_idx, path_parts in enumerate(path_parts_per_result, start=1)]


def _validate_display_names(
    display_names: list[str],
) -> None:
    if any(not display_name for display_name in display_names):
        raise ValueError("DISPLAY_NAMES entries must all be non-empty")
    name_counts: dict[str, int] = {}
    for display_name in display_names:
        name_counts[display_name] = name_counts.get(display_name, 0) + 1
    duplicated_names = sorted(name for name, count in name_counts.items() if count > 1)
    if duplicated_names:
        raise ValueError(f"DISPLAY_NAMES entries must be unique; duplicated labels: {duplicated_names}")


comparison_df = pd.DataFrame()
result_entries: list[dict[str, typing.Any]] = []
summaries = []
local_result = None
local_summary = None
main_target_entry = None
main_target_label = None

if DISPLAY_NAMES is not None and len(DISPLAY_NAMES) != len(RESULT_PATHS):
    raise ValueError(
        f"DISPLAY_NAMES has {len(DISPLAY_NAMES)} entries, expected {len(RESULT_PATHS)} to match RESULT_PATHS"
    )

normalized_result_paths = [_normalize_result_path(result_path) for result_path in RESULT_PATHS]
resolved_display_names = (
    list(DISPLAY_NAMES) if DISPLAY_NAMES is not None else _build_short_unique_path_labels(normalized_result_paths)
)
if resolved_display_names:
    _validate_display_names(resolved_display_names)

if normalized_result_paths:
    resolved_main_target_path = (
        _normalize_result_path(MAIN_TARGET_PATH) if MAIN_TARGET_PATH is not None else normalized_result_paths[0]
    )
    if resolved_main_target_path not in normalized_result_paths:
        raise ValueError(
            f"MAIN_TARGET_PATH={resolved_main_target_path} is not present in RESULT_PATHS={normalized_result_paths}"
        )
    for path_idx, result_path in enumerate(normalized_result_paths):
        display_name = resolved_display_names[path_idx]
        eval_result = pyine.evals.persistence.load_eval_result(
            result_path,
            expected_type=pyine.evals.correctness.CorrectnessEvalResult,
        )
        summary = pyine.evals.correctness.analysis.eval_result_to_summary(
            eval_result,
            subset_name=TARGET_EVAL_SUBSET_NAME,
            source_path=result_path,
            run_name=display_name,
            run_group="",
            guardrail_type_name=GUARDRAIL_TYPE_NAME,
        )
        result_entries.append(
            {
                "path": result_path,
                "display_name": display_name,
                "eval_result": eval_result,
                "summary": summary,
                "is_main_target": result_path == resolved_main_target_path,
            }
        )
        print(f"Loaded {display_name!r} from {result_path} with {len(eval_result.aggregated.per_run)} internal run(s)")
    summaries = [entry["summary"] for entry in result_entries]
    comparison_df = pyine.evals.correctness.analysis.summarize_correctness_runs_to_dataframe(summaries)
    comparison_df.insert(0, "is_main_target", [entry["is_main_target"] for entry in result_entries])
    comparison_df.insert(1, "result_path", [str(entry["path"]) for entry in result_entries])
    comparison_df.insert(2, "display_name", [entry["display_name"] for entry in result_entries])
    main_target_entry = next(entry for entry in result_entries if entry["is_main_target"])
    local_result = main_target_entry["eval_result"]
    local_summary = main_target_entry["summary"]
    main_target_label = main_target_entry["display_name"]
    print(f"Main target: {main_target_label} ({main_target_entry['path']})")
else:
    print("Populate RESULT_PATHS with one or more correctness eval pickle paths to begin.")

comparison_df  # noqa: B018 (for display purposes)

In [ ]:
# eval dataset composition across the loaded result paths
import IPython.display
import numpy as np

dataset_composition_df = pd.DataFrame()
code_type_proportions_df = pd.DataFrame()

if result_entries:
    composition_rows = []
    code_type_rows = []
    for entry in result_entries:
        aggregated = entry["eval_result"].aggregated
        class_balance = aggregated.class_balance
        sample_count = len(class_balance.per_sample_positive_rates)
        record_count = int(aggregated.split_summary.get("test_record_count", 0))
        if record_count == 0 and aggregated.attempt_records_by_key is not None:
            record_count = len(aggregated.attempt_records_by_key)
        if record_count == 0:
            record_count = sample_count
        positive_rate = class_balance.overall_positive_rate
        composition_rows.append(
            {
                "display_name": entry["display_name"],
                "result_path": str(entry["path"]),
                "sample_count": sample_count,
                "record_count": record_count,
                "positive_rate": positive_rate,
                "negative_rate": 1.0 - positive_rate,
            }
        )
        code_type_rows.append(pd.Series(class_balance.code_type_proportions, name=entry["display_name"]))
    dataset_composition_df = pd.DataFrame(composition_rows)
    code_type_proportions_df = pd.DataFrame(code_type_rows).fillna(0.0)
    if not code_type_proportions_df.empty:
        code_type_proportions_df = code_type_proportions_df.loc[
            :,
            code_type_proportions_df.mean(axis=0).sort_values(ascending=False).index,
        ]
    composition_comparable_df = dataset_composition_df.drop(columns=["result_path"]).set_index("display_name")
    identical_composition = composition_comparable_df.nunique(dropna=False).le(1).all()
    if not code_type_proportions_df.empty:
        identical_composition = identical_composition and code_type_proportions_df.nunique(dropna=False).le(1).all()
    if identical_composition:
        print("All loaded results share the same dataset composition for these summary stats.")
    else:
        print("Loaded results differ in dataset composition; compare the table and plots below.")
    print("Code type proportions are record-level and may be non-exclusive when compound code types are present.")
    IPython.display.display(
        dataset_composition_df.style.format(
            {
                "positive_rate": "{:.1%}",
                "negative_rate": "{:.1%}",
            }
        )
    )
    plot_labels = dataset_composition_df["display_name"].tolist()
    plot_positions = np.arange(len(plot_labels))
    sample_counts = dataset_composition_df["sample_count"].to_numpy()
    record_counts = dataset_composition_df["record_count"].to_numpy()
    positive_rates = dataset_composition_df["positive_rate"].to_numpy()
    negative_rates = dataset_composition_df["negative_rate"].to_numpy()
    figure_height = max(4.5, 0.75 * len(plot_labels))
    figure_width = max(18, 12 + 0.4 * max(len(code_type_proportions_df.columns), 1))
    heatmap_width = max(1.8, 0.35 * max(len(code_type_proportions_df.columns), 1))
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(figure_width, figure_height),
        gridspec_kw={"width_ratios": [1.3, 1.1, heatmap_width]},
    )
    bar_height = 0.35
    axes[0].barh(
        plot_positions - bar_height / 2,
        record_counts,
        height=bar_height,
        label="records",
        alpha=0.85,
    )
    axes[0].barh(
        plot_positions + bar_height / 2,
        sample_counts,
        height=bar_height,
        label="samples",
        alpha=0.85,
    )
    axes[0].set_yticks(plot_positions)
    axes[0].set_yticklabels(plot_labels)
    axes[0].invert_yaxis()
    axes[0].set_title("Dataset Size")
    axes[0].set_xlabel("Count")
    axes[0].legend(fontsize="small")
    axes[0].grid(axis="x", alpha=0.3)
    max_count = max([*record_counts.tolist(), *sample_counts.tolist(), 1])
    count_label_offset = max_count * 0.01
    for row_idx, record_count in enumerate(record_counts):
        axes[0].text(
            record_count + count_label_offset,
            row_idx - bar_height / 2,
            f"{int(record_count):,}",
            va="center",
            fontsize=9,
        )
    for row_idx, sample_count in enumerate(sample_counts):
        axes[0].text(
            sample_count + count_label_offset,
            row_idx + bar_height / 2,
            f"{int(sample_count):,}",
            va="center",
            fontsize=9,
        )
    axes[1].barh(plot_positions, positive_rates, label="positive", color="tab:green", alpha=0.85)
    axes[1].barh(
        plot_positions,
        negative_rates,
        left=positive_rates,
        label="negative",
        color="tab:red",
        alpha=0.7,
    )
    axes[1].set_yticks(plot_positions)
    axes[1].set_yticklabels(plot_labels)
    axes[1].invert_yaxis()
    axes[1].set_xlim(0.0, 1.0)
    axes[1].set_title("Label Balance")
    axes[1].set_xlabel("Record fraction")
    axes[1].legend(fontsize="small", loc="lower right")
    axes[1].grid(axis="x", alpha=0.3)
    for row_idx, positive_rate in enumerate(positive_rates):
        axes[1].text(
            0.5,
            row_idx,
            f"+ {positive_rate:.1%} / - {1.0 - positive_rate:.1%}",
            ha="center",
            va="center",
            fontsize=9,
        )
    if code_type_proportions_df.empty:
        axes[2].text(0.5, 0.5, "No code type data", ha="center", va="center", transform=axes[2].transAxes)
        axes[2].set_title("Code Type Proportions")
        axes[2].set_axis_off()
    else:
        heatmap_values = code_type_proportions_df.to_numpy()
        annotate_heatmap = len(plot_labels) <= 12 and len(code_type_proportions_df.columns) <= 12
        vmax = max(float(np.nanmax(heatmap_values)), 1e-9)
        image = axes[2].imshow(heatmap_values, aspect="auto", cmap="Blues", vmin=0.0, vmax=vmax)
        axes[2].set_yticks(plot_positions)
        axes[2].set_yticklabels(plot_labels)
        axes[2].set_xticks(np.arange(len(code_type_proportions_df.columns)))
        axes[2].set_xticklabels(code_type_proportions_df.columns, rotation=45, ha="right")
        axes[2].set_title("Code Type Proportions")
        if annotate_heatmap:
            for row_idx, row_values in enumerate(heatmap_values):
                for col_idx, value in enumerate(row_values):
                    if value <= 0.0:
                        continue
                    text_color = "white" if value >= vmax * 0.55 else "black"
                    axes[2].text(
                        col_idx,
                        row_idx,
                        f"{value:.0%}",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color=text_color,
                    )
        colorbar = fig.colorbar(image, ax=axes[2], fraction=0.046, pad=0.04)
        colorbar.set_label("Record fraction")
    fig.suptitle("Eval Dataset Composition Comparison")
    fig.tight_layout()
    plt.show()
else:
    print("Load one or more results before comparing dataset composition.")

dataset_composition_df  # noqa: B018 (for display purposes)

In [ ]:
# high-level threshold-free / aggregate comparison across result paths
if summaries:
    fig = pyine.evals.correctness.analysis.plot_metric_comparison(
        summaries,
        "auroc",
        title="AUROC Comparison",
    )
    plt.tight_layout()
    plt.show()

    fig = pyine.evals.correctness.analysis.plot_metric_comparison(
        summaries,
        "average_precision",
        title="Average Precision Comparison",
    )
    plt.tight_layout()
    plt.show()

    fig = pyine.evals.correctness.analysis.plot_metric_comparison(
        summaries,
        "tpr_at_fpr",
        target_fpr=TARGET_FPR,
        title=f"TPR @ FPR={TARGET_FPR}",
    )
    plt.tight_layout()
    plt.show()
else:
    print("No results found to compare")

In [ ]:
# ROC & PR curves for the main target only
if local_result is not None:
    fig = pyine.evals.correctness.analysis.plot_roc_curves(
        local_result.aggregated.per_run,
        title=f"ROC Curves ({main_target_label})",
    )
    plt.tight_layout()
    plt.show()

    fig = pyine.evals.correctness.analysis.plot_pr_curves(
        local_result.aggregated.per_run,
        title=f"Precision-Recall Curves ({main_target_label})",
    )
    plt.tight_layout()
    plt.show()
else:
    print("ROC/PR curves require at least one local CorrectnessEvalResult in RESULT_PATHS.")
    print("Populate RESULT_PATHS and optionally MAIN_TARGET_PATH.")

In [ ]:
# operating point comparison at target FPR
if summaries:
    fig = pyine.evals.correctness.analysis.plot_sample_level_metrics(
        summaries,
        TARGET_FPR,
        title=f"Sample-Level Metrics Comparison @ FPR={TARGET_FPR}",
    )
    plt.tight_layout()
    plt.show()

# detailed confusion matrix from the main target
if local_result is not None:
    main_target_run = local_result.aggregated.per_run[0]
    if TARGET_FPR in main_target_run.attempt_metrics and TARGET_FPR in main_target_run.sample_metrics:
        fig = pyine.evals.correctness.analysis.plot_operating_point_summary(
            main_target_run.attempt_metrics[TARGET_FPR],
            main_target_run.sample_metrics[TARGET_FPR],
            title=f"Operating Point Detail ({main_target_label}, FPR={TARGET_FPR})",
        )
        plt.show()
    else:
        print(f"Main target does not contain operating-point metrics at FPR={TARGET_FPR}")

In [ ]:
# category breakdown for the main target
if local_summary is not None:
    fig = pyine.evals.correctness.analysis.plot_category_breakdown(
        local_summary,
        "auroc",
        title=f"Category AUROC ({main_target_label})",
    )
    plt.tight_layout()
    plt.show()

    fig = pyine.evals.correctness.analysis.plot_category_breakdown(
        local_summary,
        "tpr",
        target_fpr=TARGET_FPR,
        title=f"Category TPR ({main_target_label}, FPR={TARGET_FPR})",
    )
    plt.tight_layout()
    plt.show()

    fig = pyine.evals.correctness.analysis.plot_category_breakdown(
        local_summary,
        "guarded_pass_rate",
        target_fpr=TARGET_FPR,
        title=f"Category Guarded Pass Rate ({main_target_label}, FPR={TARGET_FPR})",
    )
    plt.tight_layout()
    plt.show()
else:
    print("No main target summary loaded to analyze")

In [ ]:
# difficulty-conditioned analysis for the main target
if local_result is not None and local_result.aggregated.difficulty_stats is not None:
    fig = pyine.evals.correctness.analysis.plot_difficulty_analysis(
        local_result.aggregated.difficulty_stats,
        title=f"Difficulty Analysis ({main_target_label})",
    )
    plt.tight_layout()
    plt.show()
else:
    print("No difficulty stats available (requires main target result with difficulty data)")

In [ ]:
# verification cost analysis for the main target
if local_result is not None and local_result.aggregated.verification_cost_stats is not None:
    if TARGET_FPR in local_result.aggregated.verification_cost_stats:
        fig = pyine.evals.correctness.analysis.plot_cost_analysis(
            local_result.aggregated.verification_cost_stats[TARGET_FPR],
            title=f"Verification Costs ({main_target_label}, FPR={TARGET_FPR})",
        )
        plt.tight_layout()
        plt.show()
    else:
        print(f"No cost data at FPR={TARGET_FPR} for the main target")
else:
    print("No verification cost stats available (requires main target result with cost data)")

In [ ]:
# cross-result variability across the loaded result paths
if summaries and len(summaries) > 1:
    fig = pyine.evals.correctness.analysis.plot_cross_run_variability(
        summaries,
        ["auroc", "average_precision", "tpr", "guarded_pass_rate", "unsafe_slip_rate"],
        target_fpr=TARGET_FPR,
        title=f"Cross-Result Metric Variability @ FPR={TARGET_FPR}",
    )
    plt.tight_layout()
    plt.show()
else:
    print("Need multiple loaded results for variability analysis")

In [ ]:
# one-by-one attempt browser for the main target
if local_result is None:
    print("Attempt browser requires at least one local result path in RESULT_PATHS.")
else:
    main_target_run = local_result.aggregated.per_run[0]
    record_lookup = local_result.aggregated.attempt_records_by_key or {}
    print(f"main target guardrail metadata ({main_target_label}):")
    print(main_target_run.guardrail_metadata)
    attempt_rows = [row.model_dump() for row in main_target_run.attempt_records]
    print(f"Loaded {len(attempt_rows)} attempt rows from {main_target_label}")
    if not attempt_rows:
        print("No attempt rows available in this result")
    else:
        attempts_df = pd.DataFrame(attempt_rows)
        try:
            import ipywidgets as widgets
            from IPython.display import display
        except ImportError:
            widgets = None
            display = print
        if widgets is None:
            print(attempts_df.head(1).T)
        else:
            sample_filter_widget = widgets.Text(value="", description="sample_id")
            label_filter_widget = widgets.Dropdown(
                options=["all", "correct", "incorrect"],
                value="all",
                description="label",
            )
            row_idx_widget = widgets.IntSlider(
                value=0,
                min=0,
                max=max(len(attempts_df) - 1, 0),
                step=1,
                description="row_idx",
                continuous_update=False,
            )
            output_widget = widgets.Output()

            def _render_attempt(
                sample_filter: str,
                label_filter: str,
                row_idx: int,
            ) -> None:
                filtered_df = attempts_df
                if sample_filter.strip():
                    filtered_df = filtered_df[
                        filtered_df["sample_id"].astype(str).str.contains(sample_filter, na=False)
                    ]
                if label_filter == "correct":
                    filtered_df = filtered_df[filtered_df["label"]]
                elif label_filter == "incorrect":
                    filtered_df = filtered_df[~filtered_df["label"]]
                with output_widget:
                    output_widget.clear_output(wait=True)
                    if filtered_df.empty:
                        print("No rows match current filters")
                        return
                    bounded_idx = min(row_idx, len(filtered_df) - 1)
                    selected_row = filtered_df.iloc[bounded_idx]
                    sample_id = str(selected_row["sample_id"])
                    attempt_index = int(selected_row["attempt_index"])
                    draw_index = selected_row.get("draw_index")
                    scored_attempt_key = None
                    if draw_index is not None and not pd.isna(draw_index):
                        scored_attempt_key = (sample_id, attempt_index, int(draw_index))
                    record_payload = record_lookup.get(scored_attempt_key) if scored_attempt_key is not None else None
                    if record_payload is None:
                        # compatibility fallback for older pickles keyed by (sample_id, attempt_index)
                        record_payload = record_lookup.get((sample_id, attempt_index))
                    print(f"filtered row {bounded_idx + 1}/{len(filtered_df)}")
                    display(selected_row.to_frame("value"))
                    if isinstance(record_payload, dict):
                        print("\n--- raw record payload ---")
                        display(pd.Series(record_payload).to_frame("value"))
                    if isinstance(selected_row.get("attempt_metadata"), dict):
                        print("\n--- guardrail attempt metadata ---")
                        display(pd.Series(selected_row["attempt_metadata"]).to_frame("value"))

            _attempt_browser_link = widgets.interactive_output(
                _render_attempt,
                {
                    "sample_filter": sample_filter_widget,
                    "label_filter": label_filter_widget,
                    "row_idx": row_idx_widget,
                },
            )
            display(sample_filter_widget)
            display(label_filter_widget)
            display(row_idx_widget)
            display(output_widget)